# Protenix Structure Prediction

![Protenix Structure Prediction](https://proto-bio.github.io/proto-assets/images/tool/protenix/hero.png)

This notebook demonstrates multi-modal structure prediction using Protenix, an open-source reimplementation of AlphaFold3 developed by ByteDance Research. Protenix predicts the 3D structures of proteins, nucleic acids (DNA and RNA), small-molecule ligands, and their complexes. The example here predicts a protein-ligand complex: the MfnG protein bound to L-tyrosine. It covers input construction, model configuration, result inspection, visualization, and export.

In [ ]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("protenix")
display_overview("protenix")
display_docs_section("protenix", "Background")

## Available tools

In [ ]:
display_available_tools("protenix")

### `run_protenix`

Protenix is ByteDance Research's open-source reimplementation of AlphaFold3. It supports protein, DNA, RNA, and ligand inputs in a single unified model. For each complex, Protenix runs multiple diffusion samples and returns the highest-confidence prediction according to a ranking score that combines pTM, ipTM, pLDDT, and global predicted distance error. The base model with MSA integration delivers production-quality accuracy comparable to the original AlphaFold3, while mini and tiny variants offer faster inference for screening workflows.

In [ ]:
from pathlib import Path

from proto_tools import (
    Mmseqs2HomologySearchConfig,
    Chain,
    ProtenixConfig,
    ProtenixInput,
    Complex,
    run_protenix,
)

In [ ]:
# Display input API reference
display_api_reference("protenix", "input", "run_protenix")

# MfnG protein sequence (384 residues)
mfng_sequence = "MVTPEGNVSLVDESLLVGVTDEDRAVRSAHQFYERLIGLWAPAVMEAAHELGVFAALAEAPADSGELARRLDCDARAMRVLLDALYAYDVIDRIHDTNGFRYLLSAEARECLLPGTLFSLVGKFMHDINVAWPAWRNLAEVVRHGARDTSGAESPNGIAQEDYESLVGGINFWAPPIVTTLSRKLRASGRSGDATASVLDVGCGTGLYSQLLLREFPRWTATGLDVERIATLANAQALRLGVEERFATRAGDFWRGGWGTGYDLVLFANIFHLQTPASAVRLMRHAAACLAPDGLVAVVDQIVDADREPKTPQDRFALLFAASMTNTGGGDAYTFQEYEEWFTAAGLQRIETLDTPMHRILLARRATEPSAVPEGQASENLYFQ"

# L-tyrosine SMILES
tyrosine_smiles = "c1cc(ccc1C[C@@H](C(=O)O)N)O"

# Create protein-ligand complex
complex = Complex(
    chains=[
        Chain(sequence=mfng_sequence, entity_type="protein"),
        Chain(sequence=tyrosine_smiles, entity_type="ligand"),
    ]
)

# Create input
inputs = ProtenixInput(complexes=[complex])

In [ ]:
# Display config API reference
display_api_reference("protenix", "config", "run_protenix")

# Configure Protenix using the base model with MSA integration
config = ProtenixConfig(
    verbose=False,
    device="cuda",  # Change to "cpu" if no GPU available
    model_name="protenix_base_default_v1.0.0",
    use_msa=True,
    msa_search_config=Mmseqs2HomologySearchConfig(search_mode="remote"),
    num_pairformer_cycles=10,
    num_diffusion_samples=5,
    num_diffusion_steps=200,
    seeds=[0],
)

In [ ]:
# Run the prediction
result = run_protenix(inputs, config)

> NOTE: The 3D viewer below renders locally (JupyterLab, VS Code) but not in GitHub previews.

In [ ]:
# Display output API reference
display_api_reference("protenix", "output", "run_protenix")

mfng_structure = result.structures[0]

# Print confidence metrics
print(f"Number of chains:  {len(complex.chains)}")
print(f"Protein length:    {len(mfng_sequence)} residues")
print(f"Confidence score:  {mfng_structure.metrics.confidence_score:.3f}")
print(f"Average pLDDT:     {mfng_structure.metrics.avg_plddt:.3f}")
print(f"pTM score:         {mfng_structure.metrics.ptm:.3f}")
print(f"ipTM score:        {mfng_structure.metrics.iptm:.3f}")
print(f"GPDE:              {mfng_structure.metrics.get('gpde', 'N/A')}")

# Visualize predicted complex colored by pLDDT confidence
mfng_structure.visualize(style="cartoon", color_by="bfactor")

## Export Results

Predicted structures can be exported to PDB or mmCIF format for downstream analysis in molecular visualization tools such as PyMOL, ChimeraX, or VMD, and as input to molecular dynamics or further design pipelines. The B-factor column contains pLDDT confidence scores for per-residue visualization in any compatible viewer.

In [ ]:
# Create output directory
output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

# Export results to PDB files
result.export(name="mfng_ligand_complex", export_path=output_dir, file_format="pdb")